<a href="https://colab.research.google.com/github/ligafinancefiap-svg/core-finance-lib/blob/main/01_Data_Lake_Bancos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Segmento - Bancos - Ingestão e Incrementa dados da B3

Camada Bronze


In [1]:
import yfinance as yf
import pandas as pd
import os
from google.colab import drive
from datetime import datetime, timedelta

# 1. Configuração do Segmento
segmento = "bancos"
ativos = ['ITUB4.SA', 'BBDC4.SA', 'BBAS3.SA', 'SANB11.SA', 'BPAC11.SA']
caminho_arquivo = f'/content/drive/MyDrive/DATA_LAKE/01_Bronze/b3_{segmento}_brutos.csv'

drive.mount('/content/drive')
print(f"🚀 [BRONZE] Iniciando extração: {segmento.upper()}")

# 2. Lógica de Incremento
if os.path.exists(caminho_arquivo):
    df_antigo = pd.read_csv(caminho_arquivo, index_col=0, parse_dates=True)
    ultima_data = df_antigo.index.max()
    data_inicio = ultima_data + timedelta(days=1)

    if data_inicio.date() >= datetime.now().date():
        print("✅ Dados brutos já estão atualizados!")
        df_final = df_antigo
    else:
        print(f"📥 Buscando novos dados a partir de: {data_inicio.date()}")
        df_novo = yf.download(ativos, start=data_inicio, auto_adjust=True)['Close']
        df_final = pd.concat([df_antigo, df_novo]) if not df_novo.empty else df_antigo
else:
    print("📁 Criando base histórica bruta (5 anos)...")
    df_final = yf.download(ativos, period="5y", auto_adjust=True)['Close']

# 3. Salvamento na Camada Bronze
df_final.to_csv(caminho_arquivo)
print(f"💾 Arquivo salvo em 01_Bronze: b3_{segmento}_brutos.csv")

Mounted at /content/drive
🚀 [BRONZE] Iniciando extração: BANCOS
✅ Dados brutos já estão atualizados!
💾 Arquivo salvo em 01_Bronze: b3_bancos_brutos.csv
